## 핵심 포인트:
* RAGAS 없이도 가능: LangChain의 create_agent와 커스텀 프롬프트만으로 충분합니다
* 좋은 데이터가 핵심: FAQ처럼 이미 Q&A 형태로 정리된 문서가 있으면 테스트셋 생성이 훨씬 쉽습니다
* 두 가지 방식 비교: Chain 방식 vs Agent 방식을 모두 실습합니다

* 전반적인 흐르
    * FAQ 마크다운 문서 → LLM으로 Q&A 추출 → Pydantic으로 구조화 → CSV로 저장

* Chain 과 Agent (Create_agent) 방식 

| 특성 | Chain 방식 | Agent 방식 |
|------|-----------|------------|
| 구현 | `prompt \\| model` 파이프라인 | `create_agent()` 함수 |
| 유연성 | 단순하고 예측 가능 | 도구 사용 등 확장 가능 |
| 사용 시점 | 단순 변환 작업 | 복잡한 추론이 필요할 때 |

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

* GroundTruth: 개별 질문-답변 쌍
* GroundTruthList: 여러 질문-답변 쌍의 리스트

In [2]:
from pydantic import BaseModel, Field
from typing import List

class GroundTruth(BaseModel):
    question: str = Field(description="The question asked by the user.")
    answer: str = Field(description="The correct answer to the user's question.")

class GroundTruthList(BaseModel):
    ground_truths: List[GroundTruth] = Field(description="A list of ground truth question-answer pairs.")

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI
model = ChatGoogleGenerativeAI(
        model="gemini-3-flash-preview",
        temperature=0
    )
model_with_pydantic = model.with_structured_output(GroundTruthList)  

In [8]:
# prompt template 정의
from langchain_core.prompts import ChatPromptTemplate

system_prompt = """You are an AI assistant that generates a list of question-answer pairs from a given FAQ document.
Make sure to generate the pairs in Korean"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("user", "Generate question-answer pairs from the following FAQ document:\n\n {{faq_document}}")
])

In [9]:
from pathlib import Path

# 직원 복리후생 FAQ 문서 경로
# 이 파일은 이전 노트북에서 PDF를 마크다운으로 변환한 결과입니다
markdown_path = Path("documents_with_english_titles_md/employee_benefits_and_welfare_faq.md")

# 파일 내용을 문자열로 읽기 (UTF-8 인코딩)
markdown_content = markdown_path.read_text(encoding="utf-8")

In [10]:
markdown_content

'직원 복리후생 및 복지 FAQ\n=====================\n\n1. 연차 유급 휴가 관련\n-----------------\n\nQ1: 연차 휴가는 어떻게 사용하나요?\n- A: 연차 휴가는 본인 승인을 통해 자유롭게 사용할 수 있으며, 사내 HR 시스템에서 신청 가능합니다.\n\nQ2: 연차는 어떻게 계산되나요?\n- A: 근속 연수에 따라 차등 지급되며, 1년 미만은 월 1일, 1~3년은 15일, 10년 이상은 20일입니다.\n\nQ3: 미사용 연차는 어떻게 처리되나요?\n- A: 연차는 해당 연도에 사용해야 하며, 이월은 불가합니다.\n\n2. 출산 및 육아 휴가 관련\n------------------\n\nQ4: 출산 휴가는 언제부터 사용할 수 있나요?\n- A: 출산 예정일 전후로 자유롭게 사용할 수 있으며, 최대 20일 유급으로 제공됩니다.\n\nQ5: 육아 휴가는 누구에게 적용되나요?\n- A: 모든 직원이 신청할 수 있으며, 최대 1년까지 사용할 수 있으며, 첫 3개월은 급여의 50%가 지급됩니다.\n\n3. 경조 휴가 및 지원\n--------------\n\nQ6: 경조 휴가는 어떤 경우에 사용할 수 있나요?\n- A: 본인 결혼, 자녀 출생, 부모/배우자 사망 등 경조사 사유에 따라 휴가가 부여됩니다.\n\nQ7: 경조사 지원금은 어떻게 신청하나요?\n- A: HR 시스템을 통해 신청할 수 있으며, 지원금은 지정된 계좌로 지급됩니다.\n\n4. 교육 지원\n\nQ8: 외부 교육비는 어떻게 신청하나요?\n- A: 외부 교육 프로그램 신청 후 영수증을 첨부하여 HR 시스템에 신청합니다.\n\nQ9: 온라인 교육은 어떤 플랫폼을 사용할 수 있나요?\n- A: 회사 지정 플랫폼 (Inflearn, Udemy)을 자유롭게 이용할 수 있습니다.\n\n\n5. 문화 활동 및 동호회 지원\n\nQ10: 문화의 날 지원금은 어떻게 사용하나요?\n- A: 문화 활동(영화, 공연, 스포츠) 비용으로 사용 가능하며, 월 최대 50,000원 지원

### Chain 방식으로 Q&A 추출하기

In [14]:
# === Chain 방식으로 Q&A 추출 ===
# 프롬프트와 모델을 파이프라인으로 연결하여 실행합니다

import langsmith as ls
# with_structured_output: LLM이 Pydantic 모델 형태로 응답하도록 설정
# 이렇게 하면 LLM의 출력이 자동으로 GroundTruthList 객체로 변환됩니다
model_with_json = model.with_structured_output(GroundTruthList)

# Chain 생성: prompt | model_with_json
# 1. prompt: 마크다운 내용을 프롬프트에 삽입
# 2. model_with_json: 구조화된 출력(GroundTruthList)으로 응답
qna_chain = prompt | model_with_json

# Chain 실행
# invoke()에 딕셔너리를 전달하면 프롬프트의 {markdown_content}가 대체됩니다
# LangSmith 트레이싱은 선택사항입니다 (디버깅/모니터링용)
with ls.tracing_context(project_name='ai-agent-eval-study', enabled=True, tags=['chain']):
    qna_list = qna_chain.invoke({"markdown_content": markdown_content})

In [15]:
qna_list

GroundTruthList(ground_truths=[GroundTruth(question='비밀번호를 잊어버렸을 경우 어떻게 재설정하나요?', answer="로그인 화면에서 '비밀번호 찾기' 버튼을 클릭한 후 등록된 이메일 주소로 발송된 인증 링크를 통해 새 비밀번호를 설정할 수 있습니다."), GroundTruth(question='주문한 상품의 배송 상태는 어디서 확인하나요?', answer="마이페이지의 '주문 내역' 섹션에서 각 주문별 실시간 배송 상태 및 운송장 번호를 확인할 수 있습니다."), GroundTruth(question='반품 및 환불 신청은 언제까지 가능한가요?', answer='상품 수령 후 7일 이내에 단순 변심으로 인한 반품 신청이 가능하며, 상품에 결함이 있는 경우 30일 이내에 신청할 수 있습니다.')])

In [13]:
import pandas as pd

# Pydantic 모델 → 딕셔너리 → DataFrame 변환
# model_dump()는 Pydantic v2에서 dict() 대신 사용하는 메서드입니다
df = pd.DataFrame([gt.model_dump() for gt in qna_list.ground_truths])

# CSV로 저장
# utf-8-sig: Excel에서 한글이 깨지지 않도록 BOM 포함
df.to_csv("./qna_list_with_chain.csv", index=False, encoding="utf-8-sig")

print(f"Saved {len(df)} Q&A pairs to output/qna_list_with_chain.csv")
df

Saved 3 Q&A pairs to output/qna_list_with_chain.csv


,question,answer
0,비밀번호를 잊어버렸을 경우 어떻게 재설정하나요?,로그인 화면에서 '비밀번호 찾기' 버튼을 클릭한 후 등록된 이메일 주소로 발송된 인...
1,주문한 상품의 배송 상태는 어디서 확인하나요?,마이페이지의 '주문 내역' 섹션에서 각 주문별 실시간 배송 상태 및 운송장 번호를 ...
2,반품 및 환불 신청은 언제까지 가능한가요?,"상품 수령 후 7일 이내에 단순 변심으로 인한 반품 신청이 가능하며, 상품에 결함이..."


### Agent 방식으로 Q&A 추출
* Agent는 Chain과 달리 도구를 사용하거나, 여러 단계의 추론을 수행할 수 있습니다.
* 여기서는 도구 없이(tools=[]) 사용하지만, response_format으로 구조화된 출력을 지정할 수 있다는 장점이 있습니다.

* Chain과의 주요 차이점:
    - Agent는 messages 형태로 입력을 받아 대화 기록을 유지할 수 있음
    - 결과가 딕셔너리({'messages': ..., 'structured_response': ...})로 반환됨

In [16]:
from langchain.agents import create_agent

# Agent 생성
# - model: 사용할 LLM
# - tools: 에이전트가 사용할 도구들 (여기서는 빈 리스트)
# - response_format: 출력 형식을 Pydantic 모델로 지정
# - system_prompt: 에이전트의 역할 정의
agent = create_agent(
    model=model,
    tools=[],  # 이 예제에서는 도구 없이 사용
    response_format=GroundTruthList,  # 구조화된 출력
    system_prompt=system_prompt  # 위에서 정의한 시스템 프롬프트 재사용
)

In [17]:
# === Agent 실행 ===
# Agent에 마크다운 문서를 전달하여 Q&A를 추출합니다

# Agent는 messages 형태로 입력을 받습니다
# Chain과 달리 대화 기록을 유지할 수 있는 구조입니다
qna_agent = agent.invoke({
    "messages": [{"role": "user", "content": markdown_content}]
})

In [18]:
import pandas as pd

# Agent 결과에서 structured_response 추출
# Chain과 달리 Agent는 딕셔너리를 반환하므로 키로 접근합니다
structured_response = qna_agent['structured_response']

# Pydantic 모델 → DataFrame 변환
df = pd.DataFrame([gt.model_dump() for gt in structured_response.ground_truths])

# CSV로 저장
df.to_csv("./qna_list_with_agent.csv", index=False, encoding="utf-8-sig")

print(f"Saved {len(df)} Q&A pairs to output/qna_list_with_agent.csv")
df

Saved 14 Q&A pairs to output/qna_list_with_agent.csv


,question,answer
0,연차 휴가는 어떻게 사용하나요?,"연차 휴가는 본인 승인을 통해 자유롭게 사용할 수 있으며, 사내 HR 시스템에서 신..."
1,연차는 어떻게 계산되나요?,"근속 연수에 따라 차등 지급되며, 1년 미만은 월 1일, 1~3년은 15일, 10년..."
2,미사용 연차는 어떻게 처리되나요?,"연차는 해당 연도에 사용해야 하며, 이월은 불가합니다."
3,출산 휴가는 언제부터 사용할 수 있나요?,"출산 예정일 전후로 자유롭게 사용할 수 있으며, 최대 20일 유급으로 제공됩니다."
4,육아 휴가는 누구에게 적용되나요?,"모든 직원이 신청할 수 있으며, 최대 1년까지 사용할 수 있으며, 첫 3개월은 급여..."
5,경조 휴가는 어떤 경우에 사용할 수 있나요?,"본인 결혼, 자녀 출생, 부모/배우자 사망 등 경조사 사유에 따라 휴가가 부여됩니다."
6,경조사 지원금은 어떻게 신청하나요?,"HR 시스템을 통해 신청할 수 있으며, 지원금은 지정된 계좌로 지급됩니다."
7,외부 교육비는 어떻게 신청하나요?,외부 교육 프로그램 신청 후 영수증을 첨부하여 HR 시스템에 신청합니다.
8,온라인 교육은 어떤 플랫폼을 사용할 수 있나요?,"회사 지정 플랫폼 (Inflearn, Udemy)을 자유롭게 이용할 수 있습니다."
9,문화의 날 지원금은 어떻게 사용하나요?,"문화 활동(영화, 공연, 스포츠) 비용으로 사용 가능하며, 월 최대 50,000원 ..."


In [ ]:
# FAQ 문서가 동일하므로 두 방식 모두 동일한 결과로 도출하게 된다.